In [ ]:
import pandas as pd
import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA
import seaborn as sns 
from scipy.stats import pearsonr
import json
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
import plotly.express as px
from pandas.tseries.offsets import Week
from statsmodels.tsa.stattools import kpss 
import statsmodels.api as sm
import warnings
import scipy.stats as stats

### Read weather station data 

In [ ]:
tahmo_16 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00016.csv')
tahmo_98 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00098.csv')
tahmo_118 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00118.csv')

tahmo_126 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00126.csv')
tahmo_127 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00127.csv')
tahmo_313 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00313.csv')
tahmo_319 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00319.csv')
tahmo_391 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00391.csv')
tahmo_567 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00567.csv')
tahmo_647 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00647.csv')
tahmo_651 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00651.csv')

### Functions to filter weather data 

In [ ]:
def filter_columns(df):

    columns_to_keep = ['timestamp', 'lightningdistance (km)', 'lightningevents (-)', 'precipitation (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'lightningdistance (km)': 'Lightning Distance', 
        'lightningevents (-)': 'Lightning Events',
        'precipitation (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        'windgusts (m/s)': 'Wind Gusts (m/s)',
        'windspeed (m/s)': 'Wind Speed (m/s)'
    }

    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)

    return filtered_df

In [ ]:
def filter_columns_rev3(df):

    columns_to_keep = ['timestamp', 'lightningdistance (km)', 'lightningevents (-)', 'precipitation S001265 (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])

    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'lightningdistance (km)': 'Lightning Distance', 
        'lightningevents (-)': 'Lightning Events',
        'precipitation S001265 (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        'windgusts (m/s)': 'Wind Gusts (m/s)',
        'windspeed (m/s)': 'Wind Speed (m/s)'
    }
    
    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)

    return filtered_df

### 5 min re-index function 

In [ ]:
def fill_missing_timestamps(df, timestamp_col='Timestamp', freq='5min'):
    if timestamp_col in df.columns:
        df = df.copy()  # Avoid SettingWithCopyWarning

        # Convert to datetime and set index
        df[timestamp_col] = pd.to_datetime(df[timestamp_col])
        df.set_index(timestamp_col, inplace=True)

        # Generate complete timestamp range
        full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
        original_len = len(df)
        df_reindexed = df.reindex(full_range)
        df_reindexed.index.name = timestamp_col

        # Print NaN summary
        total_entries = df_reindexed.shape[0] * df_reindexed.shape[1]
        total_nans = df_reindexed.isna().sum().sum()
        nan_percent = (total_nans / total_entries) * 100 if total_entries > 0 else 0

        print(f"\nDataFrame:")
        print(f"- Original length: {original_len}")
        print(f"- After reindexing: {len(df_reindexed)} rows")
        print(f"- Total NaNs: {total_nans:,} ({nan_percent:.2f}%)")

        return df_reindexed.reset_index()
    
    else:
        print(f"Timestamp column '{timestamp_col}' not found in DataFrame.")
        return df

### Hourly resampling function  

In [ ]:
def resample_lightning_events_hourly(df, thresh=10, location='Legon'):
    df = df.copy()

    # Step 1: Convert to datetime and set index
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df.set_index('Timestamp', inplace=True)

    # Step 2: Filter for distances ≤ thresh km and non-zero events
    df_filtered = df[
        (df['Lightning Events'] != 0) &
        (df['Lightning Distance'] <= thresh)
    ].copy()

    # Step 3: Bin distances into integers (1 to thresh)
    df_filtered['Distance Bin'] = df_filtered['Lightning Distance'].apply(lambda x: int(np.ceil(x)))

    # Step 4: Group by hour and distance bin, then sum events
    df_grouped = (
        df_filtered
        .groupby([pd.Grouper(freq='H'), 'Distance Bin'])['Lightning Events']
        .sum()
        .reset_index()
        .rename(columns={'Timestamp': 'Hour'})
    )

    # Step 5: Pivot to get one row per hour, columns as distance bins
    df_pivot = df_grouped.pivot(index='Hour', columns='Distance Bin', values='Lightning Events')
    df_pivot.columns = [f"Events_{d}km" for d in df_pivot.columns]
    df_pivot = df_pivot.fillna(0).astype(int)

    # Step 6: Define fixed end hour
    desired_end_time = pd.Timestamp('2023-12-31 23:00:00')

    # Determine actual end (max of dataset or fixed end)
    start_time = df.index.min().floor('H')
    end_time = max(df.index.max().ceil('H'), desired_end_time)

    # Step 7: Reindex to include full hourly range, up to 2023-12-31 23:00:00
    full_range = pd.date_range(start=start_time, end=end_time, freq='H')
    df_pivot = df_pivot.reindex(full_range, fill_value=0)
    df_pivot.index.name = 'Hour'

    # Step 8: Add location column
    df_pivot['Location'] = location

    return df_pivot.reset_index()

### Rename & filter relevant columns from raw weather station data 

In [ ]:
# rev1 
accra_aca_df = filter_columns(tahmo_16)
g_met_hq_df = filter_columns(tahmo_98)
temasco_df = filter_columns(tahmo_118)
st_johns_df = filter_columns(tahmo_127)
safisana_df = filter_columns(tahmo_313)

nsawam_df = filter_columns(tahmo_319)
agri_impact_df = filter_columns(tahmo_391)
accra_girls_df = filter_columns(tahmo_567)
legon_df = filter_columns(tahmo_647)
madina_df = filter_columns(tahmo_651)


##rev3 
berekuso_df = filter_columns_rev3(tahmo_126)

### Legon - Lightning 

In [ ]:
# select columns of interest 
legon_df_lightning = legon_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

# conduct an hourly resampling of the lightning data 
hourly_lightning_legon = resample_lightning_events_hourly(legon_df_lightning)

# filter study period 
hourly_lightning_legon = hourly_lightning_legon.set_index('Hour').loc['2022':'2023'].reset_index()

### Madina - Lightning 

In [ ]:
madina_df_lightning = madina_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_madina = resample_lightning_events_hourly(madina_df_lightning, location = 'Madina')

hourly_lightning_madina = hourly_lightning_madina.set_index('Hour').loc['2022':'2023'].reset_index()

### G_Met - Lightning 

In [ ]:
g_met_df_lightning = g_met_hq_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_g_met = resample_lightning_events_hourly(g_met_df_lightning, location = 'G_Met')

hourly_lightning_g_met = hourly_lightning_g_met.set_index('Hour').loc['2022':'2023'].reset_index()

# add '0km' events to the '1km' event section 
hourly_lightning_g_met['Events_1km'] = hourly_lightning_g_met['Events_0km'] + hourly_lightning_g_met['Events_1km']
hourly_lightning_g_met = hourly_lightning_g_met.drop(columns = ['Events_0km'])

### Agri_Impact - Lightning 

In [ ]:
agri_impact_df_lightning = agri_impact_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_agri_impact = resample_lightning_events_hourly(agri_impact_df_lightning, location = 'Agri_Impact')

hourly_lightning_agri_impact = hourly_lightning_agri_impact.set_index('Hour').loc['2022':'2023'].reset_index()

### Berekuso - Lightning 

In [ ]:
berekuso_df_lightning = berekuso_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_berekuso = resample_lightning_events_hourly(berekuso_df_lightning, location = 'Berekuso')

hourly_lightning_berekuso = hourly_lightning_berekuso.set_index('Hour').loc['2022':'2023'].reset_index()

### Temasco - Lightning 

In [ ]:
temasco_df_lightning = temasco_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_temasco = resample_lightning_events_hourly(temasco_df_lightning, location = 'Temasco')

hourly_lightning_temasco = hourly_lightning_temasco.set_index('Hour').loc['2022':'2023'].reset_index()

### Safisana - Lightning 

In [ ]:
safisana_df_lightning = safisana_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_safisana = resample_lightning_events_hourly(safisana_df_lightning, location = 'Safisana')

hourly_lightning_safisana = hourly_lightning_safisana.set_index('Hour').loc['2022':'2023'].reset_index()

### St_Johns - Lightning 

In [ ]:
st_johns_df_lightning = st_johns_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_st_johns = resample_lightning_events_hourly(st_johns_df_lightning, location = 'St_Johns')

hourly_lightning_st_johns = hourly_lightning_st_johns.set_index('Hour').loc['2022':'2023'].reset_index()

### Accra_Aca - Lightning 

In [ ]:
accra_aca_df_lightning = accra_aca_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_accra_aca = resample_lightning_events_hourly(accra_aca_df_lightning, location = 'Accra_Aca')

hourly_lightning_accra_aca = hourly_lightning_accra_aca.set_index('Hour').loc['2022':'2023'].reset_index()

# add '0km' events to the '1km' event section 
hourly_lightning_accra_aca['Events_1km'] = hourly_lightning_accra_aca['Events_0km'] + hourly_lightning_accra_aca['Events_1km']
hourly_lightning_accra_aca = hourly_lightning_accra_aca.drop(columns = ['Events_0km'])

### Accra_Girls - Lightning 

In [ ]:
accra_girls_df_lightning = accra_girls_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_accra_girls = resample_lightning_events_hourly(accra_girls_df_lightning, location = 'Accra_Girls')

hourly_lightning_accra_girls = hourly_lightning_accra_girls.set_index('Hour').loc['2022':'2023'].reset_index()

### Nsawam - Lightning 

In [ ]:
nsawam_df_lightning = nsawam_df[['Timestamp', 'Lightning Distance', 'Lightning Events']]

hourly_lightning_nsawam = resample_lightning_events_hourly(nsawam_df_lightning, location = 'Nsawam')

hourly_lightning_nsawam = hourly_lightning_nsawam.set_index('Hour').loc['2022':'2023'].reset_index()

### Concatenate - All weather station dfs 

In [ ]:
dfs = [hourly_lightning_accra_girls, hourly_lightning_accra_aca, hourly_lightning_st_johns, hourly_lightning_berekuso, hourly_lightning_temasco, hourly_lightning_safisana, hourly_lightning_g_met, hourly_lightning_legon, hourly_lightning_madina, hourly_lightning_nsawam]

combined_lightning_df = pd.concat(dfs, ignore_index=True)

# combined_lightning_df.to_csv('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/combined_lightning_df_w_distances.csv')